# Step 3: Logistic Regression From Scratch / 第三步：从零手写逻辑回归

> **No scikit-learn! Only NumPy!** 💪
> **不用 scikit-learn！只用 NumPy！** 💪

**English:** In this notebook, we will build logistic regression with our own hands. You will see every step.

**中文：** 在这个笔记本里，我们将亲手构建逻辑回归。你会看到每一步。

## Step 0: Import NumPy and Create Data / 第0步：导入 NumPy 并创建数据

**English:** We only need NumPy. We will create simple data with 2 features.

**中文：** 我们只需要 NumPy。我们将创建有2个特征的简单数据。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility / 设置随机种子以便重复
np.random.seed(42)

# Create 100 samples with 2 features / 创建100个样本，2个特征
m = 100  # number of samples / 样本数
n = 2    # number of features / 特征数

# Features / 特征
X = np.random.randn(m, n)

# True weights and bias (we will try to learn these!) / 真实权重和偏置（我们将尝试学习它们！）
true_w = np.array([[2.0], [-1.0]])
true_b = 0.5

# Generate labels using true parameters / 用真实参数生成标签
z = np.dot(X, true_w) + true_b
y = (z > 0).astype(int)  # Simple threshold / 简单阈值

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"First 5 samples of X:\n{X[:5]}")
print(f"First 5 labels: {y[:5].T}")

## Step 1: Initialize Parameters / 第1步：初始化参数

**English:** We start with random (or zero) weights. The model will learn the correct values.

**中文：** 我们从随机（或零）权重开始。模型将学习正确的值。

---

**English:** Think of it like this: you are at the top of a mountain, and you don't know where the bottom is. You start walking!

**中文：** 这样想：你在山顶，不知道山脚在哪里。你开始走！

In [ ]:
# Initialize weights and bias to zero / 将权重和偏置初始化为零
w = np.zeros((n, 1))  # Shape: (2, 1)
b = 0.0

print(f"Initial w:\n{w}")
print(f"Initial b: {b}")

# Learning rate / 学习率
learning_rate = 0.1

# Number of iterations / 迭代次数
num_iterations = 100

print(f"Learning rate: {learning_rate}")
print(f"Number of iterations: {num_iterations}")

## Step 2: Sigmoid Function / 第2步：Sigmoid 函数

**English:** This is the function we learned before. It squeezes any number to (0, 1).

**中文：** 这是我们之前学过的函数。它将任何数字压缩到 (0, 1)。

**Formula / 公式:**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
def sigmoid(z):
    """
    Sigmoid function / Sigmoid 函数
    z: can be a number, vector, or matrix / 可以是数字、向量或矩阵
    """
    return 1 / (1 + np.exp(-z))

# Test / 测试
print(f"sigmoid(0) = {sigmoid(0):.4f}")  # Should be 0.5
print(f"sigmoid(5) = {sigmoid(5):.4f}")  # Close to 1
print(f"sigmoid(-5) = {sigmoid(-5):.4f}")  # Close to 0

## Step 3: Forward Propagation / 第3步：前向传播

**English:** This is how we make predictions. We calculate $z = w^T x + b$, then apply sigmoid.

**中文：** 这就是我们进行预测的方式。计算 $z = w^T x + b$，然后应用 sigmoid。

In [ ]:
def forward_propagation(X, w, b):
    """
    Make predictions / 进行预测
    X: features (m, n) / 特征
    w: weights (n, 1) / 权重
    b: bias (scalar) / 偏置
    """
    z = np.dot(X, w) + b
    a = sigmoid(z)
    return a

# Test with initial parameters / 用初始参数测试
a_initial = forward_propagation(X, w, b)
print(f"Predictions with initial w=0, b=0:\n{a_initial[:5].T}")
print(f"All predictions are 0.5 because z=0! / 所有预测都是0.5，因为z=0！")

## Step 4: Cost Function / 第4步：代价函数

**English:** Now we measure how wrong we are. Lower cost = better model.

**中文：** 现在我们测量我们有多错。代价越低 = 模型越好。

**Formula / 公式:**
$$J = -\frac{1}{m} \sum \left[ y \log(a) + (1-y) \log(1-a) \right]$$

In [ ]:
def compute_cost(a, y):
    """
    Compute log loss / 计算对数损失
    a: predictions (m, 1) / 预测值
    y: true labels (m, 1) / 真实标签
    """
    m = y.shape[0]
    
    # Add small number to avoid log(0) / 加小数字避免 log(0)
    epsilon = 1e-5
    
    cost = -np.sum(y * np.log(a + epsilon) + (1 - y) * np.log(1 - a + epsilon)) / m
    
    return cost

# Test with initial predictions / 用初始预测测试
cost_initial = compute_cost(a_initial, y)
print(f"Initial cost: {cost_initial:.4f}")
print(f"This is how wrong we are at the start! / 这是我们一开始的错误程度！")

## Step 5: Backward Propagation (Gradients) / 第5步：反向传播（梯度）

**English:** This calculates which direction to walk. We compute how much to change w and b.

**中文：** 这计算了朝哪个方向走。我们计算 w 和 b 应该改变多少。

**English:** Remember the mountain analogy? This is like **feeling the slope** with your feet!

**中文：** 记得山的比喻吗？这就像用脚**感觉坡度**！

In [ ]:
def backward_propagation(X, a, y):
    """
    Compute gradients / 计算梯度
    Returns dw and db / 返回 dw 和 db
    """
    m = y.shape[0]
    
    # Difference between prediction and truth / 预测与真实的差
    dz = a - y
    
    # Gradient for w / w 的梯度
    dw = np.dot(X.T, dz) / m
    
    # Gradient for b / b 的梯度
    db = np.sum(dz) / m
    
    return dw, db

# Test / 测试
dw, db = backward_propagation(X, a_initial, y)
print(f"dw shape: {dw.shape}")
print(f"dw:\n{dw}")
print(f"db: {db:.4f}")
print(f"\nThese tell us which direction to go! / 这些告诉我们该往哪个方向走！")

## Step 6: Update Parameters / 第6步：更新参数

**English:** Now we walk! We update w and b using the gradients and learning rate.

**中文：** 现在我们走路！我们用梯度和学习率更新 w 和 b。

**Formula / 公式:**
$$w := w - \alpha \cdot dw$$
$$b := b - \alpha \cdot db$$

In [ ]:
def update_parameters(w, b, dw, db, learning_rate):
    """
    Update w and b / 更新 w 和 b
    """
    w = w - learning_rate * dw
    b = b - learning_rate * db
    return w, b

# Test: update once / 测试：更新一次
w_new, b_new = update_parameters(w, b, dw, db, learning_rate)
print(f"Old w:\n{w}")
print(f"New w:\n{w_new}")
print(f"Old b: {b:.4f}")
print(f"New b: {b_new:.4f}")
print(f"\nWe took one step down the hill! / 我们朝山下走了一步！")

## Step 7: Train the Model! / 第7步：训练模型！

**English:** Now we put everything together! We repeat forward -> cost -> backward -> update, many times.

**中文：** 现在我们把所有东西放在一起！我们重复 前向→代价→反向→更新，很多次。

**English:** We will also record the cost at each step so we can watch it go down! 📉

**中文：** 我们还会记录每一步的代价，这样就能看到它下降！📉

In [ ]:
# Reset parameters / 重置参数
w = np.zeros((n, 1))
b = 0.0

# Store costs for plotting / 存储代价用于绘图
costs = []

print("Training started! / 训练开始！\n")

for i in range(num_iterations):
    # 1. Forward / 前向
    a = forward_propagation(X, w, b)
    
    # 2. Compute cost / 计算代价
    cost = compute_cost(a, y)
    costs.append(cost)
    
    # 3. Backward / 反向
    dw, db = backward_propagation(X, a, y)
    
    # 4. Update / 更新
    w, b = update_parameters(w, b, dw, db, learning_rate)
    
    # Print every 10 iterations / 每10次迭代打印一次
    if i % 10 == 0:
        print(f"Iteration {i:3d}: Cost = {cost:.4f}")

print(f"\nTraining complete! / 训练完成！")
print(f"Final cost: {costs[-1]:.4f}")
print(f"Initial cost: {costs[0]:.4f}")
print(f"Cost decreased by: {costs[0] - costs[-1]:.4f}")

## Step 8: Visualize Training / 第8步：可视化训练过程

**English:** Let's plot the cost over time. You should see it going down like walking down a hill!

**中文：** 让我们画出代价随时间的变化。你应该能看到它像下山一样下降！

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(costs, 'b-', linewidth=2)
plt.title('Cost Decreasing Over Time / 代价随时间下降', fontsize=14)
plt.xlabel('Iteration / 迭代次数')
plt.ylabel('Cost / 代价')
plt.grid(True, alpha=0.3)

# Mark start and end / 标记起点和终点
plt.plot(0, costs[0], 'ro', markersize=10, label=f'Start / 起点: {costs[0]:.4f}')
plt.plot(len(costs)-1, costs[-1], 'go', markersize=10, label=f'End / 终点: {costs[-1]:.4f}')
plt.legend()
plt.show()

print("The model is learning! / 模型正在学习！")

## Step 9: Compare with True Values / 第9步：与真实值比较

**English:** Remember we created `true_w = [[2.0], [-1.0]]` and `true_b = 0.5`? Let's see if our model learned close to these values!

**中文：** 记得我们创建了 `true_w = [[2.0], [-1.0]]` 和 `true_b = 0.5` 吗？让我们看看模型是否学到了接近这些值！

In [ ]:
print("=== Comparison / 比较 ===\n")

print("True values / 真实值:")
print(f"  true_w:\n{true_w}")
print(f"  true_b: {true_b}")

print("\nLearned values / 学习到的值:")
print(f"  w:\n{w}")
print(f"  b: {b:.4f}")

print("\nDifference / 差异:")
print(f"  w diff: {np.abs(w - true_w).flatten()}")
print(f"  b diff: {abs(b - true_b):.4f}")

print("\nThe model learned the correct direction! / 模型学到了正确的方向！")

## Step 10: Make Predictions / 第10步：进行预测

**English:** Now use the trained model to predict on new data!

**中文：** 现在用训练好的模型在新数据上预测！

In [ ]:
# Predict on training data / 在训练数据上预测
a_final = forward_propagation(X, w, b)
predictions = (a_final >= 0.5).astype(int)

# Calculate accuracy / 计算准确率
accuracy = np.mean(predictions == y) * 100

print(f"Accuracy: {accuracy:.2f}%")
print(f"\nFirst 10 predictions vs true labels / 前10个预测 vs 真实标签:")
print(f"Predicted: {predictions[:10].flatten()}")
print(f"Actual:    {y[:10].flatten()}")

## Congratulations! / 恭喜！🎉

**English:** You just built logistic regression from scratch! You wrote:

✅ Sigmoid function
✅ Cost function
✅ Gradient descent
✅ Training loop

**中文：** 你刚刚从零构建了逻辑回归！你写了：

✅ Sigmoid 函数
✅ 代价函数
✅ 梯度下降
✅ 训练循环

---

**English:** Now when you use `scikit-learn`, you know exactly what happens inside!

**中文：** 现在当你用 `scikit-learn` 时，你完全知道里面发生了什么！

## Exercise: Try Different Learning Rates / 练习：尝试不同的学习率

**English:** Go back to Step 7 and try changing `learning_rate` to:
- 0.01 (very small)
- 0.5 (very big)
- 1.0 (too big?)

Watch what happens to the cost!

**中文：** 回到第7步，尝试改变 `learning_rate` 为：
- 0.01（很小）
- 0.5（很大）
- 1.0（太大？）

观察代价会发生什么！


啊啊钱啊啊